In [1]:
!pip install "mlflow>=3.5.0" pyngrok --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.5/788.5 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.7 MB/s eta 0:00:00


In [2]:
import os
import mlflow

# All tracking data lives only inside this Colab VM
os.environ["MLFLOW_TRACKING_URI"] = "file:///content/mlruns"

EXPERIMENT_NAME = "colab_mlflow_demo"
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/01/28 16:48:40 INFO mlflow.tracking.fluent: Experiment with name 'colab_mlflow_demo' does not exist. Creating a new experiment.


Tracking URI: file:///content/mlruns
Experiment: colab_mlflow_demo


In [3]:
from pyngrok import ngrok
from getpass import getpass
import os, time

# 1) Disable MLflow security middleware (safe enough for Colab teaching)
os.environ["MLFLOW_SERVER_DISABLE_SECURITY_MIDDLEWARE"] = "true"

# 2) Kill anything that might already be using port 5000 (best-effort)
get_ipython().system_raw("fuser -k 5000/tcp 2>/dev/null || true")

# 3) Start MLflow tracking server in the background
#    (this serves both the API and the UI on port 5000)
get_ipython().system_raw(
    "mlflow server "
    "--host 0.0.0.0 "
    "--port 5000 "
    "--backend-store-uri /content/mlruns &"
)

# Give the server a few seconds to start
time.sleep(5)

# 4) Ask for ngrok token (hidden input so it’s not visible in the notebook)
NGROK_AUTH_TOKEN = getpass("🔑 Paste your ngrok auth token: ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Clean up any old tunnels
ngrok.kill()

# 5) Create the tunnel to port 5000
public_url = ngrok.connect(5000, "http")
print("🔗 MLflow UI URL:", public_url)

🔑 Paste your ngrok auth token: ··········
🔗 MLflow UI URL: NgrokTunnel: "https://philip-intercorporate-autographically.ngrok-free.dev" -> "http://localhost:5000"


In [4]:
import time
import math
import random
import mlflow
import matplotlib.pyplot as plt
import numpy as np

def run_toy_experiment(learning_rate: float, num_epochs: int, run_name: str = None):
    """
    Simple toy experiment where 'loss' decays over epochs with some noise.
    Logs:
      - Params: learning_rate, num_epochs
      - Metrics: loss per epoch
      - Artifact: a PNG plot of the loss curve
    """
    if run_name is None:
        run_name = f"lr={learning_rate}_epochs={num_epochs}"

    with mlflow.start_run(run_name=run_name):
        # Log hyperparameters
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("num_epochs", num_epochs)

        losses = []

        # Fake training loop
        base_loss = 1.0 / learning_rate  # Just to change the scale a bit

        for epoch in range(num_epochs):
            # Fake loss: exponential decay + some random noise
            noise = random.uniform(-0.05, 0.05)
            loss = base_loss * math.exp(-0.2 * epoch) + noise
            losses.append(loss)

            # Log metric to MLflow
            mlflow.log_metric("loss", loss, step=epoch)

            # Simulate training time
            time.sleep(0.05)

        # Create and log a loss curve plot as an artifact
        epochs = np.arange(num_epochs)

        plt.figure()
        plt.plot(epochs, losses)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"Loss curve (lr={learning_rate})")

        os.makedirs("plots", exist_ok=True)
        plot_path = f"plots/loss_lr_{learning_rate}.png"
        plt.savefig(plot_path)
        plt.close()

        # Log the artifact
        mlflow.log_artifact(plot_path, artifact_path="loss_plots")

        print(f"Finished run: {run_name}")

In [5]:
# A few different hyperparameter configurations
configs = [
    (0.1, 20),
    (0.01, 30),
    (0.05, 25),
]

for lr, epochs in configs:
    run_toy_experiment(learning_rate=lr, num_epochs=epochs)

Finished run: lr=0.1_epochs=20
Finished run: lr=0.01_epochs=30
Finished run: lr=0.05_epochs=25


In [6]:
import pandas as pd

runs_df = mlflow.search_runs()
runs_df[[
    "run_id",
    "tags.mlflow.runName",
    "params.learning_rate",
    "params.num_epochs",
    "metrics.loss"
]].head()

,run_id,tags.mlflow.runName,params.learning_rate,params.num_epochs,metrics.loss
0,5869d6fb7074416c97343bcf99409ee4,lr=0.05_epochs=25,0.05,25,0.190447
1,bf7abd8370134b99b21cefa2cb531dc8,lr=0.01_epochs=30,0.01,30,0.278336
2,c19621b67ce74915b82125105595be13,lr=0.1_epochs=20,0.1,20,0.176099
